In [ ]:
# Step 0: Install imbalanced-learn if not already
!pip install imbalanced-learn --quiet

import pandas as pd
from imblearn.over_sampling import SMOTE
from google.colab import files

# -------------------------------
# Step 1: Upload your file
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# -------------------------------
# Step 2: Read Excel
data = pd.read_excel(filename)

# Clean column names
data.columns = data.columns.str.strip().str.replace('\n',' ').str.replace('\r',' ')

# -------------------------------
# Step 3: Map Yes/No to numeric for key columns
yes_no_cols = ['Chronic_Disease','Prescription_Meds','Alt_Use','Informed_Doctor']
for col in yes_no_cols:
    if col in data.columns:
        data[col] = data[col].map({'Yes':1, 'No':0})

# -------------------------------
# Step 4: Fill missing numeric columns with 0
numeric_cols = ['Age', 'Eff_AltMed', 'Alt_Safety', 'Alt_Trust']
for col in numeric_cols:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce').fillna(0)

# -------------------------------
# Step 5: Fill all object / categorical NaNs with 'Missing'
for col in data.columns:
    if data[col].dtype == 'object' or str(data[col].dtype) == 'category':
        data[col] = data[col].fillna('Missing')

# -------------------------------
# Step 6: One-hot encode all categorical/object columns for SMOTE
categorical_cols = data.select_dtypes(include=['object','category']).columns
data_encoded = pd.get_dummies(data, columns=categorical_cols, drop_first=False)

# -------------------------------
# Step 7: Separate features and target
X = data_encoded.drop('Alt_Use', axis=1)
y = data_encoded['Alt_Use']

# Fill any remaining NaNs
X = X.fillna(0)

# -------------------------------
# Step 8: Apply SMOTE
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# Create balanced dataframe
balanced_data = pd.concat([pd.DataFrame(X_res, columns=X.columns),
                           pd.Series(y_res, name='Alt_Use')], axis=1)

# -------------------------------
# Step 9: Recombine one-hot columns back to original categories

def recombine_one_hot(df, prefix, mapping=None, default='Missing'):
    """Recombine one-hot encoded columns starting with prefix into a single column."""
    cols = [c for c in df.columns if c.startswith(prefix)]
    if not cols:
        return df  # nothing to combine
    def get_category(row):
        selected = [c for c in cols if row[c] > 0]
        if not selected:
            return default
        cat = selected[0].replace(prefix,'')
        if mapping and cat in mapping:
            return mapping[cat]
        return cat
    df[prefix.rstrip('_')] = df.apply(get_category, axis=1)
    df = df.drop(columns=cols)
    return df

# Gender
balanced_data = recombine_one_hot(
    balanced_data,
    'Gender_',
    mapping={'Male':'Male','Others / Prefer not to say':'Other'},
    default='Female'
)

# Education
balanced_data = recombine_one_hot(
    balanced_data,
    'Education_',
    mapping={'Primary':'Primary','Secondary':'Secondary','Undergraduate':'Undergraduate'},
    default='Missing'
)

# Income
balanced_data = recombine_one_hot(
    balanced_data,
    'Income_',
    default='Missing'
)

# Alt_Reason
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Reason_',
    default='Missing'
)

# Alt_Type
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Type_',
    default='Missing'
)

# Alt_Expense
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Expense_',
    default='Missing'
)

# Alt_Disease
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Disease_',
    default='Missing'
)

# Alt_Frequency
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Frequency_',
    default='Missing'
)

# Alt_Replace
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Replace_',
    default='Missing'
)

# Alt_Info
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Info_',
    default='Missing'
)

# Alt_Not_Reason
balanced_data = recombine_one_hot(
    balanced_data,
    'Alt_Not_Reason_',
    default='Missing'
)

# -------------------------------
# Step 10: Save final dataset
balanced_data.to_excel("balanced_primary_data_final.xlsx", index=False)
files.download("balanced_primary_data_final.xlsx")

print("Balanced dataset reconstructed and saved as 'balanced_primary_data_final.xlsx'")

Saving primary_data.xlsx to primary_data (25).xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Balanced dataset reconstructed and saved as 'balanced_primary_data_final.xlsx'
